# 04 — Testar QLoRA em 4 bits na GPU

Execute depois de concluir os notebooks 00, 02 e 03 e de obter o diagnóstico do 01. Este caderno testa o carregamento real do Qwen3-8B e **uma atualização dos adaptadores LoRA**, usando somente dados de treino. O resultado permite avaliar a memória da sessão antes de implementar o treinamento completo.

O perfil adicional usa pesos NF4 de 4 bits, cálculo FP16, rank 8, módulos `q_proj`/`v_proj`, microlote 1 e contexto de até 1.024 tokens. Ele é descrito em `config/perfil_qlora_teste_1024.json`. O teste usa uma atualização com acumulação 1; a proposta de treino completo conserva acumulação 16 e será avaliada separadamente.

**Use uma sessão GPU dedicada a este notebook.** Uma sessão nova evita que modelos ou tensores de outro caderno ocupem a memória. Não carregue o E5 junto com o Qwen. Os dados e relatórios ficam no Drive; os pesos são mantidos no armazenamento temporário da sessão.

A quantização reduz a memória do modelo na GPU, mas o primeiro download ainda pode transferir aproximadamente 16 GB de pesos originais. Reserve pelo menos 22 GiB livres no armazenamento temporário. A viabilidade do treino completo e a qualidade científica ainda dependerão dos próximos testes.

**Revisão 2 — contexto de 1.024 tokens.** O teste anterior esgotou a memória da GPU durante o passo de treinamento. Os 160 exemplos de treino têm de 208 a 744 tokens com este prompt e alvo, portanto continuam completos. A redução diminui o padding e o consumo esperado de ativações/logits. O novo teste confirma se a memória é suficiente nesta sessão.

Preserve a sessão atual se ela ainda estiver conectada e a GPU estiver livre: o cache em `/content` pode ser reutilizado. Não é necessário excluir o ambiente ou repetir os notebooks 00–03.


In [ ]:
from pathlib import Path
from google.colab import drive
import os

# Defina o cache antes de qualquer biblioteca Hugging Face ser importada.
os.environ["HF_HOME"] = "/content/hf_cache_llm_eidt_br"
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Replicação LLM EIDT IC"  # @param {type:"string"}
raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
ponto_de_montagem = Path("/content/drive").resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError("Selecione a pasta completa do projeto dentro do Drive montado.")
print("Pasta selecionada:", raiz)
print("Cache temporário:", os.environ["HF_HOME"])


## 1. Conferir os arquivos da atualização

Copie ao Drive o novo notebook, o perfil QLoRA, o script e o índice de hashes fornecidos nesta atualização. O programa também confere a configuração base e os hashes da seleção, do pré-processamento e do arquivo de treino. Se houver divergência, confira a cópia antes de continuar.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib import metadata
import json
import re
import subprocess
import sys
import tempfile

indice = raiz / "SHA256SUMS.txt"
assinaturas = {}
for linha in indice.read_text(encoding="utf-8-sig").splitlines():
    item = re.fullmatch(r"([0-9a-f]{64})  (.+)", linha)
    if item:
        nome = item.group(2)
        if nome in assinaturas:
            raise ValueError(f"Hash duplicado no índice para {nome}.")
        assinaturas[nome] = item.group(1)

arquivos_requeridos = (
    "requirements.txt",
    "config/experimento.json",
    "config/perfil_qlora_teste_1024.json",
    "scripts/testar_qlora_gpu.py",
)
for nome in arquivos_requeridos:
    caminho = raiz / nome
    if not caminho.is_file() or sha256(caminho.read_bytes()).hexdigest() != assinaturas.get(nome):
        raise ValueError(f"Falha de integridade em {nome}. Recopie o arquivo e o índice da mesma atualização.")
    print("Conferido:", nome)

arquivo_requisitos = raiz / "requirements.txt"
perfil_relativo = "config/perfil_qlora_teste_1024.json"
pasta_relatorios = raiz / "resultados/etapa_1/piloto_br_200"
pasta_relatorios.mkdir(parents=True, exist_ok=True)
inicio_notebook = datetime.now(timezone.utc)
relatorio_preparacao = pasta_relatorios / f"preparacao_qlora_notebook_{inicio_notebook:%Y%m%dT%H%M%S%fZ}.json"
preparacao = {
    "inicio_utc": inicio_notebook.isoformat(),
    "estado": "preparando_ambiente",
    "cache_temporario": os.environ["HF_HOME"],
    "sha256_arquivos": {nome: assinaturas[nome] for nome in arquivos_requeridos},
}

def salvar_preparacao():
    preparacao["atualizado_utc"] = datetime.now(timezone.utc).isoformat()
    temporario = relatorio_preparacao.with_suffix(".tmp")
    temporario.write_text(json.dumps(preparacao, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporario.replace(relatorio_preparacao)

salvar_preparacao()


## 2. Conferir as dependências

A instalação respeita as versões do projeto e conserva as versões de PyTorch e NumPy fornecidas pelo Colab. O teste de modelo será iniciado em um processo Python novo, que usa os pacotes instalados. Por isso, este notebook não pede um reinício após a instalação. Uma falha de compatibilidade interrompe a preparação e deve ser examinada antes do teste.

In [ ]:
try:
    versoes_protegidas = {nome: metadata.version(nome) for nome in ("torch", "numpy")}
    preparacao["versoes_protegidas"] = versoes_protegidas
    desejadas = {}
    for linha in arquivo_requisitos.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.split("#", 1)[0].strip()
        if linha:
            if not re.fullmatch(r"[A-Za-z0-9_.-]+==[^\s]+", linha):
                raise ValueError("Os requisitos devem usar versões exatas no formato pacote==versão.")
            nome, versao = linha.split("==", 1)
            desejadas[nome] = versao
    divergentes = []
    for nome, esperada in desejadas.items():
        try:
            atual = metadata.version(nome)
        except metadata.PackageNotFoundError:
            atual = None
        if atual != esperada:
            divergentes.append(nome)
    preparacao["pacotes_que_requeriam_instalacao"] = divergentes
    if divergentes:
        with tempfile.TemporaryDirectory(dir="/content") as pasta_temporaria:
            restricoes = Path(pasta_temporaria) / "base_colab.txt"
            plano_pip = Path(pasta_temporaria) / "operacoes_pip.json"
            restricoes.write_text(
                "".join(f"{nome}=={versao}\n" for nome, versao in versoes_protegidas.items()),
                encoding="utf-8",
            )
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                 "--progress-bar", "off", "-c", str(restricoes), "-r", str(arquivo_requisitos),
                 "--report", str(plano_pip)],
                check=True,
            )
            operacoes = json.loads(plano_pip.read_text(encoding="utf-8"))["install"]
            preparacao["pacotes_alterados"] = sorted(item["metadata"]["name"] for item in operacoes)
    else:
        preparacao["pacotes_alterados"] = []
    for nome, anterior in versoes_protegidas.items():
        if metadata.version(nome) != anterior:
            raise RuntimeError(f"A versão protegida de {nome} mudou. Interrompa e confira o ambiente.")
    preparacao["versoes_requisitos"] = {nome: metadata.version(nome) for nome in desejadas}
    if preparacao["versoes_requisitos"] != desejadas:
        raise RuntimeError("Alguma dependência ainda diverge da versão fixada.")
    preparacao["estado"] = "dependencias_conferidas"
    print("Dependências conferidas. PyTorch e NumPy nativos preservados.")
except Exception as erro:
    preparacao["estado"] = "falha_na_instalacao"
    preparacao["tipo_erro"] = type(erro).__name__
    raise
finally:
    salvar_preparacao()


## 3. Conferir GPU, RAM e armazenamento temporário

A verificação ocorre em um processo separado que termina antes do carregamento do modelo. O teste real medirá o consumo de GPU. Se não houver GPU, escolha uma no menu **Ambiente de execução → Alterar tipo de ambiente de execução**. Se faltar espaço temporário, use uma sessão nova e conserve os dados já salvos no Drive.

In [ ]:
if preparacao["estado"] != "dependencias_conferidas":
    raise RuntimeError("Conclua a instalação antes de conferir os recursos.")

codigo_recursos = r"""
import json, platform, shutil
from pathlib import Path
import torch

def memoria_sistema():
    dados = {}
    arquivo = Path('/proc/meminfo')
    if arquivo.exists():
        valores = {}
        for linha in arquivo.read_text().splitlines():
            chave, valor = linha.split(':', 1)
            if chave in {'MemTotal', 'MemAvailable'}:
                valores[chave] = int(valor.strip().split()[0]) * 1024
        dados = {chave: round(valor / 2**30, 2) for chave, valor in valores.items()}
    return dados

resultado = {
    'python': platform.python_version(), 'torch': str(torch.__version__),
    'cuda_pytorch': torch.version.cuda, 'cuda_disponivel': torch.cuda.is_available(),
    'disco_livre_gib': round(shutil.disk_usage('/content').free / 2**30, 2),
    'ram_gib': memoria_sistema(),
}
if resultado['cuda_disponivel']:
    livre, total = torch.cuda.mem_get_info()
    resultado.update(gpu=torch.cuda.get_device_name(0),
                     capacidade_cuda=list(torch.cuda.get_device_capability(0)),
                     vram_livre_gib=round(livre / 2**30, 2),
                     vram_total_gib=round(total / 2**30, 2))
print(json.dumps(resultado, ensure_ascii=False))
"""
try:
    verificacao = subprocess.run([sys.executable, "-c", codigo_recursos],
                                 capture_output=True, text=True, check=True)
    recursos = json.loads(verificacao.stdout.strip().splitlines()[-1])
    preparacao["recursos"] = recursos
    print(json.dumps(recursos, ensure_ascii=False, indent=2))
    if not recursos["cuda_disponivel"]:
        raise RuntimeError("GPU não disponível. Selecione uma GPU no Colab e execute novamente.")
    if recursos["capacidade_cuda"][0] < 6:
        raise RuntimeError("Esta GPU não atende ao perfil NF4 preparado. Envie o relatório para revisão.")
    if recursos["disco_livre_gib"] < 22:
        raise RuntimeError("É necessário ter pelo menos 22 GiB livres em /content antes do download. Use uma sessão nova.")
    preparacao["estado"] = "pronto_para_teste_real"
except Exception as erro:
    preparacao["estado"] = "recursos_requerem_atencao"
    preparacao["tipo_erro"] = type(erro).__name__
    detalhe = getattr(erro, "stderr", None)
    if detalhe:
        print(detalhe[-4000:])
    raise
finally:
    salvar_preparacao()


## 4. Carregar o Qwen3-8B e executar o teste curto

Esta célula faz o download na revisão fixada, audita comprimentos dos exemplos de treino e testa uma atualização LoRA. O teste usa perda somente na resposta CPC. O alvo CPC mantém a definição de proxy registrada no protocolo.

A primeira execução pode demorar durante o download. O programa imprime as fases e registra um relatório no Drive, inclusive se ocorrer falha. A ausência de memória ou gradientes inválidos interrompe o teste e exige revisão da configuração. Nenhum adaptador de produção é salvo por este diagnóstico.

In [ ]:
if preparacao["estado"] != "pronto_para_teste_real":
    raise RuntimeError("Conclua a conferência de recursos antes de carregar o modelo.")

relatorios_anteriores = set(pasta_relatorios.glob("qlora_teste_*.json"))
resultado = subprocess.run(
    [sys.executable, "-u", str(raiz / "scripts/testar_qlora_gpu.py"),
     "--raiz", str(raiz), "--perfil", perfil_relativo],
    cwd=raiz, env=os.environ.copy(), check=False,
)
relatorios_novos = sorted(
    set(pasta_relatorios.glob("qlora_teste_*.json")) - relatorios_anteriores,
    key=lambda caminho: caminho.name,
)
if not relatorios_novos:
    raise RuntimeError("O processo terminou sem relatório novo. Envie as últimas mensagens desta célula.")
relatorio_teste = relatorios_novos[-1]
relatorio = json.loads(relatorio_teste.read_text(encoding="utf-8"))
passo = relatorio.get("passo_executado", {})
memoria = relatorio.get("memoria", {})
ultima_memoria = memoria.get("apos_passo", memoria.get("na_falha", {}))
resumo = {
    "estado": relatorio.get("estado"),
    "perfil": relatorio.get("perfil", {}).get("id"),
    "etapa": relatorio.get("etapa"),
    "etapa_falha": relatorio.get("etapa_falha"),
    "loss": passo.get("loss", relatorio.get("loss_forward")),
    "pico_alocado_gib": ultima_memoria.get("pico_alocado_gib"),
    "pico_reservado_gib": ultima_memoria.get("pico_reservado_gib"),
    "duracao_segundos": relatorio.get("duracao_segundos"),
    "erro": relatorio.get("erro"),
}
print("\nRESUMO DO TESTE — copie este bloco")
print(json.dumps(resumo, ensure_ascii=False, indent=2))
print("Relatório completo:", relatorio_teste)
print("Preparação do ambiente:", relatorio_preparacao)
if resultado.returncode != 0 or resumo["estado"] != "teste_qlora_aprovado":
    raise RuntimeError(
        f"Teste não aprovado: {resumo['estado']}. "
        "Envie o resumo acima; não repita a mesma configuração automaticamente."
    )


## 5. Enviar o resultado para conferência

Envie o resumo final do teste e o caminho do relatório. Os arquivos já ficam guardados no Drive. Se uma célula falhar, envie a mensagem de erro e o relatório que tiver sido produzido.

A aprovação confirma o carregamento e uma atualização no perfil testado, com a memória observada nesta sessão. Ela não mede a qualidade da predição CPC, não substitui o treino de três épocas nem valida extração, alinhamento ou indicadores. O próximo passo será definido a partir desse resultado e da cobertura dos rótulos já registrada no piloto.